# Radar Target Classifier — PYNQ-Z2 FPGA

In [ ]:
import numpy as np, time, json
from pynq import Overlay
from IPython.display import display, HTML, clear_output
ol = Overlay('cnn_overlay.bit')
cnn = None
for name in ol.ip_dict:
    if 'cnn' in name.lower(): cnn = getattr(ol, name); break
led_gpio = ol.axi_gpio_0
led_gpio.write(0x04, 0x0)
d = np.load('pynq_demo_data.npz', allow_pickle=True)
demo_iq = d['demo_iq_real'] + 1j * d['demo_iq_imag']
demo_labels = d['demo_labels']
demo_classes = d['demo_classes']
X_test = d['X_test']; y_test = d['y_test']
with open('quant_params.json') as f: qp = json.load(f)
ACT_MAX=qp['ACT_MAX']; M1=qp['M1']; M2=qp['M2']; M3=qp['M3']; RSHIFT=qp['REQUANT_SHIFT']
print('Overlay loaded')


In [ ]:
def extract_features(iq_vec):
    iq=iq_vec.reshape(5,256); mag=np.abs(iq); center=iq[2,:]
    ps=np.abs(np.fft.fft(center)[:128]).reshape(32,4).mean(1)
    mv=np.var(mag,axis=0).reshape(16,16).mean(1)
    st=[]
    for i in range(5):
        m=mag[i,:]; st+=[np.mean(m),np.std(m),np.max(m)-np.min(m)]
    ph=np.mean(np.abs(np.diff(np.angle(center))))
    feat=np.concatenate([ps,mv,np.array(st),[ph]])
    mn,mx=feat.min(),feat.max()
    if mx-mn>1e-10: feat=(feat-mn)/(mx-mn)*255.0
    return feat

def classify_fpga(features):
    feat=np.clip(np.round(features),0,255).astype(np.uint8)
    for i in range(16):
        w=int(feat[i*4])|(int(feat[i*4+1])<<8)|(int(feat[i*4+2])<<16)|(int(feat[i*4+3])<<24)
        cnn.write(0x40+i*4,w)
    cnn.write(0x00,0x01)
    for _ in range(10000):
        if cnn.read(0x00)&0x02: break
    return int(cnn.read(0x10))
print('Functions ready')


## Live Classification

In [ ]:
CN=['Drone','Bird','Human','Reflector']
CC=['#ff4444','#4499ff','#44dd44','#ffaa00']
CI=['\u2708\uFE0F','\U0001F426','\U0001F6B6','\U0001F4D0']
results=[]
log=''
ok_count=0
total_us=0

for i in range(len(demo_iq)):
    # LED sweep while scanning
    for s in range(2):
        for led in range(4):
            led_gpio.write(0x00, 1<<led)
            time.sleep(0.03)
    led_gpio.write(0x00, 0)

    # Classify on FPGA
    feat=extract_features(demo_iq[i])
    t0=time.time()
    pred=classify_fpga(feat)
    us=int((time.time()-t0)*1e6)
    tc=int(demo_classes[i])
    ok=pred==tc
    lb=str(demo_labels[i])
    if ok: ok_count+=1
    total_us+=us
    n=i+1
    results.append(dict(id=i,label=lb,true_cls=tc,pred=pred,correct=ok,time_us=us))

    # Light result LED
    led_gpio.write(0x00, 1<<pred)

    # Build log row
    ck='\u2714' if ok else '\u2718'
    bc='#00ff88' if ok else '#ff4444'
    bg='#0a1a10' if ok else '#1a0808'
    log='<div style="display:flex;padding:4px 8px;margin:2px 0;border-radius:4px;font-size:11px;border-left:3px solid '+bc+';background:'+bg+'">'
    log+='<span style="width:28px;color:#445">#'+str(i).zfill(2)+'</span>'
    log+='<span style="width:130px;color:#778">'+lb+'</span>'
    log+='<span style="width:90px;font-weight:bold;color:'+CC[pred]+'">'+CI[pred]+' '+CN[pred]+'</span>'
    log+='<span style="width:55px;color:#4499ff;text-align:right">'+str(us)+'us</span>'
    log+='<span style="width:22px;text-align:center;color:'+bc+'">'+ck+'</span></div>\n'+log

    # LED dots
    ld=''
    for led in range(4):
        on=led==pred
        lc=CC[led] if on else '#1a1a2a'
        sh='0 0 10px '+CC[led]+',0 0 20px '+CC[led] if on else 'none'
        ld+='<div style="width:18px;height:18px;border-radius:50%;background:'+lc+';border:1px solid #333;box-shadow:'+sh+';display:inline-block;margin:0 6px"></div>'

    acc=int(100*ok_count/n)
    avg=total_us//n

    # Build GUI
    h='<div style="font-family:Courier New,monospace;background:#060a12;color:#ccc;padding:15px;border-radius:10px">'

    # Header
    h+='<div style="text-align:center;margin-bottom:12px">'
    h+='<div style="font-size:14px;color:#00ff88;letter-spacing:3px;font-weight:bold">RADAR TARGET CLASSIFICATION</div>'
    h+='<div style="font-size:9px;color:#445;margin-top:3px">PYNQ-Z2 | ZYNQ-7020 | 77 GHz FMCW | 1D CNN INT8</div></div>'

    # Stats
    h+='<div style="display:flex;gap:8px;margin-bottom:12px">'
    h+='<div style="flex:1;background:#0d1520;border:1px solid #1a2a3a;border-radius:8px;padding:8px;text-align:center"><div style="font-size:20px;font-weight:bold;color:#00ff88">'+str(acc)+'%</div><div style="font-size:7px;color:#445">ACCURACY</div></div>'
    h+='<div style="flex:1;background:#0d1520;border:1px solid #1a2a3a;border-radius:8px;padding:8px;text-align:center"><div style="font-size:20px;font-weight:bold;color:#4499ff">'+str(avg)+'us</div><div style="font-size:7px;color:#445">AVG SPEED</div></div>'
    h+='<div style="flex:1;background:#0d1520;border:1px solid #1a2a3a;border-radius:8px;padding:8px;text-align:center"><div style="font-size:20px;font-weight:bold;color:#ffaa00">'+str(n)+'/'+str(len(demo_iq))+'</div><div style="font-size:7px;color:#445">SCANNED</div></div>'
    h+='<div style="flex:1;background:#0d1520;border:1px solid #1a2a3a;border-radius:8px;padding:8px;text-align:center"><div style="font-size:20px;font-weight:bold;color:#ff4444">212/220</div><div style="font-size:7px;color:#445">DSP</div></div>'
    h+='</div>'

    # Current target
    h+='<div style="background:#0a1018;border:2px solid '+CC[pred]+';border-radius:10px;padding:18px;text-align:center;margin-bottom:12px">'
    h+='<div style="font-size:9px;color:#445;letter-spacing:2px">CURRENT TARGET</div>'
    h+='<div style="font-size:50px;margin:6px 0">'+CI[pred]+'</div>'
    h+='<div style="font-size:24px;font-weight:bold;color:'+CC[pred]+';letter-spacing:3px">'+CN[pred]+'</div>'
    h+='<div style="font-size:10px;color:#556;margin-top:4px">Signature: '+lb+' | True: '+CN[tc]+'</div>'
    h+='<div style="font-size:12px;color:'+bc+';margin-top:5px;font-weight:bold">'+ck+(' CORRECT' if ok else ' MISMATCH')+'</div>'
    h+='<div style="margin-top:8px">'+ld+'</div>'
    h+='<div style="font-size:7px;color:#334;margin-top:3px">LD0 &nbsp;&nbsp; LD1 &nbsp;&nbsp; LD2 &nbsp;&nbsp; LD3</div>'
    h+='</div>'

    # Detection log
    h+='<div style="font-size:10px;color:#00ff88;letter-spacing:2px;margin-bottom:4px">DETECTION LOG</div>'
    h+='<div style="max-height:150px;overflow-y:auto;border:1px solid #1a2a3a;border-radius:6px;padding:3px">'+log+'</div>'

    h+='<div style="text-align:center;margin-top:8px;font-size:7px;color:#223">ERTEKIN &amp; GUNEY (2025) | KARLSSON DATASET | KTH</div>'
    h+='</div>'

    clear_output(wait=True)
    display(HTML(h))

    # Blink LED
    time.sleep(0.3)
    for _ in range(3):
        led_gpio.write(0x00, 0); time.sleep(0.06)
        led_gpio.write(0x00, 1<<pred); time.sleep(0.06)
    time.sleep(0.6)

led_gpio.write(0x00, 0)


## Spectrograms

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import spectrogram as spec
fig=plt.figure(figsize=(16,10))
fig.patch.set_facecolor('#060a12')
fig.suptitle('MICRO-DOPPLER SIGNATURES',color='#00ff88',fontsize=16,fontweight='bold',fontfamily='monospace',y=0.98)
cs={0:[],1:[],2:[],3:[]}
for idx in range(len(demo_iq)):
    c=int(demo_classes[idx])
    if len(cs[c])<2: cs[c].append(idx)
pi=0
CN=['Drone','Bird','Human','CR']
for cl in range(4):
    for j,idx in enumerate(cs[cl]):
        pi+=1
        ax=fig.add_subplot(2,4,pi)
        ax.set_facecolor('#0a1018')
        center=demo_iq[idx].reshape(5,256)[2,:]
        fr,t,Sxx=spec(center,fs=17000,nperseg=64,noverlap=48)
        ax.pcolormesh(t*1000,fr/1000,10*np.log10(np.abs(Sxx)+1e-10),cmap='hot',shading='gouraud')
        feat=extract_features(demo_iq[idx])
        pred=classify_fpga(feat)
        okk=pred==cl
        col='#00ff88' if okk else '#ff4444'
        for s in ax.spines.values(): s.set_color(col); s.set_linewidth(2)
        ax.set_title(CN[cl],color=col,fontsize=10,fontweight='bold',fontfamily='monospace')
        ax.tick_params(colors='#444',labelsize=6)
        ax.set_xlabel('Time (ms)',color='#666',fontsize=7)
        ax.set_ylabel('Freq (kHz)',color='#666',fontsize=7)
plt.tight_layout(rect=[0,0,1,0.95])
plt.savefig('spectrograms.png',dpi=150,facecolor='#060a12')
plt.show()


## Speed Analysis

In [ ]:
times=[r['time_us'] for r in results]
fig,ax=plt.subplots(figsize=(8,4))
fig.patch.set_facecolor('#060a12')
ax.set_facecolor('#0a1018')
ax.hist(times,bins=15,color='#00ff88',edgecolor='#060a12')
ax.axvline(np.mean(times),color='#ff4444',linewidth=2,linestyle='--',label=f'Mean: {int(np.mean(times))}us')
ax.set_xlabel('Inference Time (us)',color='white')
ax.set_ylabel('Count',color='white')
ax.set_title('FPGA INFERENCE LATENCY',color='#00ff88',fontfamily='monospace')
ax.tick_params(colors='white')
ax.legend(facecolor='#0a1018',edgecolor='#1a2a3a',labelcolor='white')
for s in ax.spines.values(): s.set_color('#1a2a3a')
plt.tight_layout()
plt.savefig('speed_hist.png',dpi=150,facecolor='#060a12')
plt.show()


## Per-Class Accuracy

In [ ]:
CN=['Drone','Bird','Human','CR']
CC=['#ff4444','#4499ff','#44dd44','#ffaa00']
fig,ax=plt.subplots(figsize=(8,4))
fig.patch.set_facecolor('#060a12')
ax.set_facecolor('#0a1018')
accs=[]
for c in range(4):
    t=sum(1 for r in results if r['true_cls']==c)
    o=sum(1 for r in results if r['true_cls']==c and r['correct'])
    accs.append(100*o/max(1,t))
bars=ax.bar(CN,accs,color=CC,width=0.6,edgecolor='#060a12')
for bar,val in zip(bars,accs):
    ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,f'{val:.0f}%',ha='center',color='white',fontweight='bold')
ax.set_ylim(0,110)
ax.set_ylabel('Accuracy %',color='white')
ax.set_title('PER-CLASS ACCURACY',color='#00ff88',fontfamily='monospace')
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_color('#1a2a3a')
plt.tight_layout()
plt.savefig('class_accuracy.png',dpi=150,facecolor='#060a12')
plt.show()


## Confusion Matrix (500 test samples)

In [ ]:
N=min(500,len(X_test))
cm=np.zeros((4,4),dtype=int)
print(f'Testing {N} samples...')
for i in range(N):
    cm[int(y_test[i]),classify_fpga(X_test[i])]+=1
    if (i+1)%100==0: print(f'  {i+1}/{N}')
fig,ax=plt.subplots(figsize=(7,6))
fig.patch.set_facecolor('#060a12')
ax.set_facecolor('#0a1018')
im=ax.imshow(cm,cmap='YlGn')
for i in range(4):
    for j in range(4):
        c='white' if cm[i,j]>cm.max()*0.5 else '#00ff88'
        pct=int(100*cm[i,j]/max(1,cm[i].sum()))
        ax.text(j,i,f'{cm[i,j]}\n{pct}%',ha='center',va='center',fontsize=12,fontweight='bold',color=c,fontfamily='monospace')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(CN,color='white'); ax.set_yticklabels(CN,color='white')
ax.set_xlabel('Predicted',color='white'); ax.set_ylabel('True',color='white')
acc=100*cm.trace()/cm.sum()
ax.set_title(f'FPGA CNN - {acc:.1f}% Accuracy',color='#00ff88',fontfamily='monospace')
plt.colorbar(im,shrink=0.8)
plt.tight_layout()
plt.savefig('confusion.png',dpi=150,facecolor='#060a12')
plt.show()


## FPGA Resource Utilization

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(12,4))
fig.patch.set_facecolor('#060a12')
resources=[('DSP',212,220),('LUT',23627,53200),('BRAM',36,140)]
for ax,(name,used,total) in zip(axes,resources):
    ax.set_facecolor('#060a12')
    pct=100*used/total
    ax.pie([used,total-used],colors=['#00ff88','#1a2a3a'],startangle=90,wedgeprops=dict(width=0.35,edgecolor='#060a12'))
    ax.text(0,0,f'{pct:.0f}%',ha='center',va='center',fontsize=20,fontweight='bold',color='#00ff88',fontfamily='monospace')
    ax.set_title(f'{name}\n{used}/{total}',color='white',fontsize=11,fontfamily='monospace')
plt.suptitle('FPGA RESOURCE UTILIZATION',color='#00ff88',fontsize=14,fontfamily='monospace',y=1.02)
plt.tight_layout()
plt.savefig('resources.png',dpi=150,facecolor='#060a12',bbox_inches='tight')
plt.show()


## Radar Signal Analysis

In [ ]:
CN=['Drone','Bird','Human','CR']
CC=['#ff4444','#4499ff','#44dd44','#ffaa00']
fig,axes=plt.subplots(1,4,figsize=(14,4))
fig.patch.set_facecolor('#060a12')
fig.suptitle('RANGE PROFILES BY CLASS',color='#00ff88',fontsize=14,fontfamily='monospace',y=1.02)
for cls in range(4):
    ax=axes[cls]
    ax.set_facecolor('#0a1018')
    idxs=[i for i in range(len(demo_iq)) if int(demo_classes[i])==cls]
    for idx in idxs[:5]:
        iq=demo_iq[idx].reshape(5,256)
        rp=np.abs(iq).mean(axis=1)
        ax.plot(range(1,6),rp,color=CC[cls],alpha=0.5,linewidth=1.5)
    ax.set_title(CN[cls],color=CC[cls],fontsize=11,fontfamily='monospace')
    ax.set_xlabel('Range Cell',color='white',fontsize=9)
    ax.set_ylabel('Magnitude',color='white',fontsize=9)
    ax.tick_params(colors='white',labelsize=7)
    for s in ax.spines.values(): s.set_color('#1a2a3a')
plt.tight_layout()
plt.savefig('range_profiles.png',dpi=150,facecolor='#060a12',bbox_inches='tight')
plt.show()


In [ ]:
fig,axes=plt.subplots(1,4,figsize=(14,4))
fig.patch.set_facecolor('#060a12')
fig.suptitle('DOPPLER SPECTRA BY CLASS',color='#00ff88',fontsize=14,fontfamily='monospace',y=1.02)
for cls in range(4):
    ax=axes[cls]
    ax.set_facecolor('#0a1018')
    idxs=[i for i in range(len(demo_iq)) if int(demo_classes[i])==cls]
    for idx in idxs[:5]:
        center=demo_iq[idx].reshape(5,256)[2,:]
        doppler=np.abs(np.fft.fftshift(np.fft.fft(center)))
        freqs=np.fft.fftshift(np.fft.fftfreq(256,d=1/17000))
        ax.plot(freqs,10*np.log10(doppler+1e-10),color=CC[cls],alpha=0.4,linewidth=1)
    ax.set_title(CN[cls],color=CC[cls],fontsize=11,fontfamily='monospace')
    ax.set_xlabel('Doppler (Hz)',color='white',fontsize=9)
    ax.set_ylabel('Power (dB)',color='white',fontsize=9)
    ax.tick_params(colors='white',labelsize=7)
    for s in ax.spines.values(): s.set_color('#1a2a3a')
plt.tight_layout()
plt.savefig('doppler_spectra.png',dpi=150,facecolor='#060a12',bbox_inches='tight')
plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
fig.patch.set_facecolor('#060a12')
ax.set_facecolor('#0a1018')
data_pc=[]
for cls in range(4):
    idxs=[i for i in range(len(demo_iq)) if int(demo_classes[i])==cls]
    strengths=[20*np.log10(np.abs(demo_iq[i]).mean()+1e-10) for i in idxs]
    data_pc.append(strengths)
bp=ax.boxplot(data_pc,labels=CN,patch_artist=True)
for patch,color in zip(bp['boxes'],CC):
    patch.set_facecolor(color+'44')
    patch.set_edgecolor(color)
for item in ['whiskers','caps','medians']:
    for line in bp[item]: line.set_color('white')
for flier in bp['fliers']: flier.set_markeredgecolor('#556')
ax.set_ylabel('Signal Strength (dB)',color='white',fontsize=11)
ax.set_title('RADAR RETURN STRENGTH BY TARGET',color='#00ff88',fontsize=13,fontfamily='monospace')
ax.tick_params(colors='white')
for s in ax.spines.values(): s.set_color('#1a2a3a')
plt.tight_layout()
plt.savefig('signal_strength.png',dpi=150,facecolor='#060a12')
plt.show()


In [ ]:
fig,axes=plt.subplots(1,4,figsize=(14,4))
fig.patch.set_facecolor('#060a12')
fig.suptitle('PHASE VARIATION (MICRO-DOPPLER)',color='#00ff88',fontsize=14,fontfamily='monospace',y=1.02)
for cls in range(4):
    ax=axes[cls]
    ax.set_facecolor('#0a1018')
    idxs=[i for i in range(len(demo_iq)) if int(demo_classes[i])==cls]
    for idx in idxs[:3]:
        center=demo_iq[idx].reshape(5,256)[2,:]
        phase=np.unwrap(np.angle(center))
        phase_diff=np.diff(phase)
        t=np.arange(len(phase_diff))/17000*1000
        ax.plot(t,phase_diff,color=CC[cls],alpha=0.5,linewidth=1)
    ax.set_title(CN[cls],color=CC[cls],fontsize=11,fontfamily='monospace')
    ax.set_xlabel('Time (ms)',color='white',fontsize=9)
    ax.set_ylabel('Phase Rate',color='white',fontsize=9)
    ax.tick_params(colors='white',labelsize=7)
    for s in ax.spines.values(): s.set_color('#1a2a3a')
plt.tight_layout()
plt.savefig('phase_variation.png',dpi=150,facecolor='#060a12',bbox_inches='tight')
plt.show()


In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
fig.patch.set_facecolor('#060a12')
ax.set_facecolor('#0a1018')
feat_matrix=[]
for cls in range(4):
    idxs=[i for i in range(len(demo_iq)) if int(demo_classes[i])==cls]
    for idx in idxs[:5]:
        feat_matrix.append(extract_features(demo_iq[idx]))
feat_matrix=np.array(feat_matrix)
im=ax.imshow(feat_matrix.T,aspect='auto',cmap='inferno',interpolation='nearest')
ax.set_xlabel('Sample',color='white',fontsize=11)
ax.set_ylabel('Feature Index',color='white',fontsize=11)
ax.set_title('FEATURE VECTORS ACROSS CLASSES',color='#00ff88',fontsize=13,fontfamily='monospace')
ax.tick_params(colors='white')
for i in range(1,4):
    ax.axvline(i*5-0.5,color='#00ff88',linewidth=1,linestyle='--')
ax.set_xticks([2,7,12,17])
ax.set_xticklabels(CN,color='white')
plt.colorbar(im,ax=ax,shrink=0.8)
plt.tight_layout()
plt.savefig('features_heatmap.png',dpi=150,facecolor='#060a12')
plt.show()
